# DATA

## EXTRACCION DE DATOS
Para el desarrollo de este proyecto se recurrió a dos tipos de fuentes. Por un lado, las pistas de audio utilizadas para construir el dataset fueron obtenidas de Free Music Archive (https://freemusicarchive.org), plataforma de distribución musical que ofrece canciones bajo licencias Creative Commons, permitiendo su uso libre en contextos académicos e investigativos sin restricciones de derechos de autor. Las canciones fueron seleccionadas y organizadas manualmente por género musical, garantizando un mínimo de 50 pistas por categoría.

## PROCESAMIENTO Y MODELAMIENTO DE LOS DATOS

### Librerias 
Librosa: es una librería especializada en análisis de audio y música que permite cargar archivos de sonido y extraer características acústicas. Es el núcleo del proceso de extracción de features en este proyecto.

NumPy: es la librería fundamental para computación numérica en Python. Proporciona estructuras de datos tipo array de alto rendimiento y operaciones matemáticas vectorizadas, siendo la base sobre la que trabajan la mayoría de librerías científicas incluyendo Librosa.

Pandas es la librería estándar para manipulación y análisis de datos tabulares en Python. Permite construir, limpiar y transformar dataframes, que es la estructura utilizada para almacenar y exportar el dataset final en formato CSV.

Os es un módulo nativo de Python que permite interactuar con el sistema operativo, específicamente con el sistema de archivos. En este proyecto se utiliza para recorrer las carpetas de géneros, listar los archivos de audio disponibles y construir las rutas de acceso a cada canción.

### Extracción de Features 
El código implementa un pipeline de extracción automática de características acústicas a partir de archivos de audio organizados por género musical. Para cada canción se carga la pista completa y se extraen tres segmentos de 30 segundos correspondientes al inicio, la mitad y el final de la pista. De cada segmento se calculan 27 features: siete características acústicas globales (tempo, spectral centroid, spectral bandwidth, rolloff, zero crossing rate, RMS energy y chroma STFT) y 20 coeficientes MFCC que representan el timbre de la señal. Cada segmento procesado queda registrado como una fila independiente en el dataset, junto con su etiqueta de género y el nombre del archivo de origen. El resultado final es un archivo CSV estructurado y listo para la etapa de modelado.

## CODIGO PARA SEGMENTOS DE 30s

In [ ]:
"""
import librosa
import numpy as np
import pandas as pd
import os
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

def extraerfeatures(y, sr):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    
    features = {
        'tempo':               librosa.beat.tempo(y=y, sr=sr)[0],
        'spectral_centroid':   librosa.feature.spectral_centroid(y=y, sr=sr).mean(),
        'spectral_bandwidth':  librosa.feature.spectral_bandwidth(y=y, sr=sr).mean(),
        'rolloff':             librosa.feature.spectral_rolloff(y=y, sr=sr).mean(),
        'zero_crossing_rate':  librosa.feature.zero_crossing_rate(y).mean(),
        'rms':                 librosa.feature.rms(y=y).mean(),
        'chroma_stft':         chroma.mean(),
        'onset_strength_mean': onset_env.mean(),
        'onset_strength_std':  onset_env.std(),
        'pitch_variance':      chroma.var(),
        'spectral_flux_std':   onset_env.std() 
    }
    
    for i, coef in enumerate(mfccs):
        features[f'mfcc{i+1}_mean'] = coef.mean()
        features[f'mfcc{i+1}_std']  = coef.std()
        
    return features

# --- CONFIGURACIÓN ---
filas = []
carpeta_raiz = 'data/raw/songs' 

# Contadores para el reporte
stats = {"procesados": 0, "error_lectura": 0, "muy_cortos": 0}

generos = [d for d in os.listdir(carpeta_raiz) if os.path.isdir(os.path.join(carpeta_raiz, d))]

for genero in generos:
    carpeta_genero = os.path.join(carpeta_raiz, genero)
    archivos_genero = [f for f in os.listdir(carpeta_genero) if f.endswith(('.mp3', '.wav'))]
    
    for archivo in tqdm(archivos_genero, desc=f"Procesando {genero}"):
        ruta = os.path.join(carpeta_genero, archivo)
        try:
            # Cargamos el audio
            y, sr = librosa.load(ruta, sr=None)
            duracion = librosa.get_duration(y=y, sr=sr)

            # LÓGICA DE SEGMENTACIÓN DINÁMICA: Rescata canciones cortas
            puntos_segmento = []
            
            if duracion >= 85:
                # Caso estándar: 3 segmentos
                puntos_segmento = [
                    (0, "inicio"), 
                    (int(duracion/2)-15, "mitad"), 
                    (int(duracion)-30, "final")
                ]
            elif duracion >= 60:
                # Rescate: 2 segmentos
                puntos_segmento = [(0, "inicio"), (int(duracion)-30, "final")]
            elif duracion >= 30:
                # Rescate mínimo: 1 segmento
                puntos_segmento = [(0, "unico")]
            else:
                stats["muy_cortos"] += 1
                continue

            for inicio_seg, nombre_seg in puntos_segmento:
                inicio_p = int(inicio_seg * sr)
                fin_p    = int((inicio_seg + 30) * sr)
                segmento = y[inicio_p:fin_p]

                # Margen de seguridad de 29s para evitar errores de redondeo
                if len(segmento) >= int(29 * sr): 
                    features = extraerfeatures(segmento, sr)
                    features['label'] = genero
                    features['song_id'] = archivo
                    features['segment_type'] = nombre_seg
                    filas.append(features)
            
            stats["procesados"] += 1

        except Exception:
            stats["error_lectura"] += 1
            continue

# CREACIÓN DEL DATAFRAME
df = pd.DataFrame(filas)

# --- REPORTE FINAL ---
print(f"\n--- RESUMEN DEL PROCESO ACTUALIZADO ---")
print(f"Canciones procesadas con éxito: {stats['procesados']}")
print(f"Canciones saltadas (menos de 30s): {stats['muy_cortos']}")
print(f"Canciones con error de archivo: {stats['error_lectura']}")
print(f"Total de segmentos en el DataFrame: {df.shape[0]}")
print(f"\nDistribución por género:\n{df['label'].value_counts()}")

"""

## POSIBLE CODIGO PARA SEGMENTOS DE 15s

Toca revisar aca a ver si hay que hacer cambios

In [ ]:
import os
import warnings
from typing import Any, Dict, List
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

# Desactivar advertencias de librerías externas para mantener limpia la consola
warnings.filterwarnings('ignore')


def extraer_features_audio(y: np.ndarray, sr: int) -> Dict[str, Any]:
    """Extractor completo de características musicales para ventanas de 15 segundos.
    
    Calcula las 37 variables físicas de audio iniciales sin aplicar filtros
    previos, garantizando la integridad total del set de datos original.

    Args:
        y (np.ndarray): Vector de la señal de audio.
        sr (int): Tasa de muestreo (Sample Rate).

    Returns:
        Dict[str, Any]: Diccionario con las 37 características físicas extraídas.
    """
    # Extracción de espectrogramas y vectores base
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    
    # Estimación del tempo (BPM)
    tempo_estimado = float(librosa.beat.tempo(y=y, sr=sr)[0])

    # Construcción del bloque de características base (Variables 0 a 10)
    features = {
        'tempo':               tempo_estimado,
        'spectral_centroid':   float(librosa.feature.spectral_centroid(y=y, sr=sr).mean()),
        'spectral_bandwidth':  float(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()),
        'rolloff':             float(librosa.feature.spectral_rolloff(y=y, sr=sr).mean()),
        'zero_crossing_rate':  float(librosa.feature.zero_crossing_rate(y).mean()),
        'rms':                 float(librosa.feature.rms(y=y).mean()),
        'chroma_stft':         float(chroma.mean()),
        'onset_strength_mean': float(onset_env.mean()),
        'onset_strength_std':  float(onset_env.std()),
        'pitch_variance':      float(chroma.var()),
        'spectral_flux_std':   float(onset_env.std()) 
    }

    # Extracción iterativa de los 13 coeficientes MFCC (Variables 11 a 36)
    for i, coef in enumerate(mfccs):
        num_mfcc = i + 1
        features[f'mfcc{num_mfcc}_mean'] = float(coef.mean())
        features[f'mfcc{num_mfcc}_std']  = float(coef.std())
        
    return features


def procesar_audio_ventanas(
    ruta_audio: str, 
    duracion_vent_sec: float = 15.0, 
    solapamiento_sec: float = 7.5
) -> List[Dict[str, Any]]:
    """Segmenta un archivo de audio mediante una ventana deslizable (Sliding Window)

    para multiplicar el volumen de registros disponibles.

    Args:
        ruta_audio (str): Ruta física del archivo de audio (.mp3 o .wav).
        duracion_vent_sec (float): Duración del segmento en segundos (15s).
        solapamiento_sec (float): Paso de desplazamiento de la ventana (7.5s).

    Returns:
        List[Dict[str, Any]]: Lista de diccionarios con los fragmentos procesados.
    """
    segmentos_extraidos = []
    
    # Cargar el archivo de sonido respetando sus propiedades originales
    y, sr = librosa.load(ruta_audio, sr=None)
    duracion_total = librosa.get_duration(y=y, sr=sr)

    # Omitir pistas que no alcancen la duración mínima de la ventana operativa
    if duracion_total < duracion_vent_sec:
        return []

    # Conversión matemática de unidades de tiempo a muestras binarias
    muestras_ventana = int(duracion_vent_sec * sr)
    muestras_paso = int(solapamiento_sec * sr)

    inicio_muestra = 0
    contador_segmento = 1

    # Desplazamiento continuo de la ventana a lo largo de la canción
    while inicio_muestra + muestras_ventana <= len(y):
        fin_muestra = inicio_muestra + muestras_ventana
        chunk_audio = y[inicio_muestra:fin_muestra]

        # Procesar el fragmento actual si cumple con el margen de seguridad estable
        if len(chunk_audio) >= int(duracion_vent_sec * 0.98 * sr):
            features = extraer_features_audio(chunk_audio, sr)
            features['segment_type'] = f"ventana_{contador_segmento}"
            segmentos_extraidos.append(features)

        # Avanzar el puntero según el paso definido por el solapamiento
        inicio_muestra += muestras_paso
        contador_segmento += 1

    return segmentos_extraidos


# --- PIPELINE PRINCIPAL DE EXTRACCIÓN DE DATOS ---

CARPETA_RAIZ = 'data/raw/songs'
filas_dataset = []

# Diccionario de control de calidad y auditoría del proceso
stats = {"procesados": 0, "error_lectura": 0, "muy_cortos": 0}

if not os.path.exists(CARPETA_RAIZ):
    raise FileNotFoundError(f"La ruta especificada '{CARPETA_RAIZ}' no existe.")

# Escaneo de subdirectorios correspondientes a los géneros musicales
generos = [d for d in os.listdir(CARPETA_RAIZ) if os.path.isdir(os.path.join(CARPETA_RAIZ, d))]

for genero in generos:
    carpeta_genero = os.path.join(CARPETA_RAIZ, genero)
    archivos_audio = [f for f in os.listdir(carpeta_genero) if f.endswith(('.mp3', '.wav'))]
    
    for archivo in tqdm(archivos_audio, desc=f"Procesando {genero.upper()}"):
        ruta_completa = os.path.join(carpeta_genero, archivo)
        
        try:
            # Segmentación dinámica en bloques de 15 segundos con overlap del 50%
            segmentos = procesar_audio_ventanas(
                ruta_completa, 
                duracion_vent_sec=15.0, 
                solapamiento_sec=7.5
            )

            if not segmentos:
                stats["muy_cortos"] += 1
                continue

            # Inyección de metadatos finales (Variables 37 a 39)
            for segmento in segmentos:
                segmento['label'] = genero
                segmento['song_id'] = archivo
                filas_dataset.append(segmento)
            
            stats["procesados"] += 1

        except Exception:
            stats["error_lectura"] += 1
            continue

# Consolidación final del DataFrame Estructurado
df_musical = pd.DataFrame(filas_dataset)

# --- REPORTE EJECUTIVO DE CONTROL ---
print("\n" + "="*50)
print("       REPORTE DE EJECUCIÓN: NUEVO PIPELINE V3")
print("="*50)
print(f"Canciones procesadas con éxito:       {stats['procesados']}")
print(f"Canciones descartadas (< 15s):        {stats['muy_cortos']}")
print(f"Archivos corruptos o con error:       {stats['error_lectura']}")
print("-"*50)
print(f"Total de registros generados (Filas): {df_musical.shape[0]}")
print(f"Total de características (Columnas):   {df_musical.shape[1]} (40 esperadas)")
print("-"*50)
print("Distribución de fragmentos por género:")
print(df_musical['label'].value_counts())
print("="*50)

# Exportar a CSV listo para el entrenamiento de Optuna
# df_musical.to_csv('datav3_15s.csv', index=False)

## RESULTADOS

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='label', palette='viridis')
plt.title('Distribución de Segmentos por Género')
plt.xlabel('Género Musical')
plt.ylabel('Cantidad de Muestras')
plt.xticks(rotation=45)
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Normalizar los datos
features = df.select_dtypes(include=[np.number]).columns
x = StandardScaler().fit_transform(df[features])

# 2. Aplicar PCA
pca = PCA(n_components=2)
components = pca.fit_transform(x)

# 3. Graficar
plt.figure(figsize=(10, 8))
sns.scatterplot(x=components[:,0], y=components[:,1], hue=df['label'], alpha=0.7)
plt.title('Visualización de Géneros en 2D (PCA)')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

In [ ]:
df

In [ ]:
df.to_csv('datav1.csv', index=False)